# NB08 — Evaluation: Compare Methods & Declare Winner — RTX 5090

Evaluate all trained agents (**PPO**, **SAC**, **Residual-SAC-best-β**)
on **200 deterministic episodes** each. Compute **95 % bootstrap CI**
(50 K resamples), **Welch's t-test**, and **Cohen's d** effect sizes.
Produce comparison tables and plots. **Declare the winner** for NB09.

| Metric | Spec |
|--------|------|
| Episodes | 200 per method (seeds 0-199) |
| Bootstrap | 50,000 resamples, 95 % CI |
| Statistical Test | Welch's t (unequal variance) |
| Effect Size | Cohen's d |


## Objectives

1. Load all 3 trained models (from NB05 / NB06 / NB07-best-β).
2. Evaluate each on **200 deterministic episodes** (same seed sequence).
3. Compute **50 K bootstrap 95 % CI** for mean reward & success rate.
4. Run **Welch's t-test** on every pair of methods.
5. Compute **Cohen's d** effect size for every pair.
6. Build comparison table & stat_tests artifact.
7. Declare winner (highest mean_reward; tie-break by success_rate).
8. Produce bar charts with CI, violin plots, success rate plots.


## Resources

| Resource | Requirement | Notes |
|----------|-------------|-------|
| GPU | Optional | CPU OK for eval |
| RAM | 8+ GB | 200 eps × 3 methods |
| Runtime | ~1-2 hours | 600 episodes total |


## Imports & Setup


In [ ]:
import sys, os, json, random, itertools
from pathlib import Path

import numpy as np
import torch
import gymnasium as gym
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats  # Welch's t-test

from stable_baselines3 import PPO, SAC

import mani_skill.envs
from mani_skill.utils.wrappers.gymnasium import CPUGymWrapper

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.envs import UnitreeG1PlaceAppleInBowlFullBodyEnv

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "NB08"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


## Configuration


In [ ]:
CFG = {
    "env_id":           "UnitreeG1PlaceAppleInBowlFullBody-v1",
    "control_mode":     "pd_joint_delta_pos",
    "obs_mode":         "state",
    "eval_episodes":    200,
    "max_steps_per_ep": 1000,
    "bootstrap_n":      50_000,
    "ci_level":         0.95,
}

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

with open(ARTIFACTS_DIR / "nb08_config.json", "w") as f:
    json.dump(CFG, f, indent=2)
print("Config:", json.dumps(CFG, indent=2))


## Step 1 — Load All Models


In [ ]:
# Load best beta info
with open(PROJECT_ROOT / "artifacts" / "NB07" / "best_beta.json") as f:
    best_beta_info = json.load(f)

best_beta = best_beta_info["best_beta"]
best_model_name = best_beta_info["best_model"].replace(".zip", "")

models = {
    "PPO": PPO.load(str(PROJECT_ROOT / "artifacts" / "NB05" / "ppo_apple")),
    "SAC": SAC.load(str(PROJECT_ROOT / "artifacts" / "NB06" / "sac_apple")),
    "Residual-SAC": SAC.load(str(PROJECT_ROOT / "artifacts" / "NB07" / best_model_name)),
}

print(f"Loaded models: {list(models.keys())}")
print(f"Residual-SAC best β = {best_beta}")


## Step 2 — BaseController for Residual-SAC Evaluation


In [ ]:
# Re-define for evaluation (same as NB07)
def heuristic_policy(obs, env):
    action = np.zeros(env.action_space.shape[0], dtype=np.float32)
    arm_start, arm_end = 13, 23
    action[arm_start:arm_end] = np.random.uniform(-0.1, 0.1, arm_end - arm_start) * 0.5
    return action

class BaseController:
    def __init__(self, env, alpha=0.3):
        self.env = env
        self.alpha = alpha
        self._prev_action = None

    def get_action(self, obs):
        raw = heuristic_policy(obs, self.env)
        if self._prev_action is None:
            self._prev_action = raw.copy()
        smoothed = self.alpha * raw + (1 - self.alpha) * self._prev_action
        self._prev_action = smoothed.copy()
        return smoothed

    def reset(self):
        self._prev_action = None

class ResidualActionWrapper(gym.ActionWrapper):
    def __init__(self, env, base_controller, beta=0.5):
        super().__init__(env)
        self.base_controller = base_controller
        self.beta = beta
        self._current_obs = None

    def reset(self, **kwargs):
        self.base_controller.reset()
        result = self.env.reset(**kwargs)
        self._current_obs = result[0] if isinstance(result, tuple) else result
        return result

    def step(self, action):
        result = self.env.step(action)
        if isinstance(result, tuple):
            self._current_obs = result[0]
        return result

    def action(self, residual_action):
        base_action = self.base_controller.get_action(self._current_obs)
        final = base_action + self.beta * residual_action
        return np.clip(final, self.action_space.low, self.action_space.high)

print("✅ BaseController + ResidualActionWrapper ready")


## Step 3 — Evaluate All Methods (200 episodes each)


In [ ]:
def evaluate_model(model, env, n_episodes, max_steps):
    """Run n deterministic episodes, return per-episode results."""
    results = []
    for ep in range(n_episodes):
        obs, info = env.reset(seed=ep)
        ep_reward, ep_steps = 0.0, 0
        for step in range(max_steps):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            ep_reward += float(reward)
            ep_steps += 1
            if terminated or truncated:
                break
        results.append({
            "episode": ep, "total_reward": ep_reward,
            "steps": ep_steps, "success": bool(info.get("success", False)),
        })
    return results

all_results = {}

for method_name, model in models.items():
    print(f"\nEvaluating {method_name} ({CFG['eval_episodes']} episodes)...")

    if method_name == "Residual-SAC":
        raw_env = gym.make(CFG["env_id"], num_envs=1, obs_mode=CFG["obs_mode"],
                           control_mode=CFG["control_mode"], render_mode="rgb_array")
        raw_env = CPUGymWrapper(raw_env)
        base_ctrl = BaseController(raw_env, alpha=0.3)
        eval_env = ResidualActionWrapper(raw_env, base_ctrl, beta=best_beta)
    else:
        eval_env = gym.make(CFG["env_id"], num_envs=1, obs_mode=CFG["obs_mode"],
                            control_mode=CFG["control_mode"], render_mode="rgb_array")
        eval_env = CPUGymWrapper(eval_env)

    results = evaluate_model(model, eval_env, CFG["eval_episodes"],
                             CFG["max_steps_per_ep"])
    all_results[method_name] = results
    eval_env.close()

    rews = [r["total_reward"] for r in results]
    succs = [r["success"] for r in results]
    print(f"  mean_reward={np.mean(rews):.4f} ± {np.std(rews):.4f}, "
          f"success_rate={np.mean(succs):.2%}")

# Save per-episode data
rows = []
for method, results in all_results.items():
    for r in results:
        rows.append({"method": method, **r})
eval_df = pd.DataFrame(rows)
eval_df.to_csv(ARTIFACTS_DIR / "eval_200ep.csv", index=False)
print(f"\nSaved: eval_200ep.csv ({len(rows)} rows)")


## Step 4 — Bootstrap 95 % Confidence Intervals (50 K resamples)


In [ ]:
def bootstrap_ci(data, stat_fn=np.mean, n_bootstrap=50_000, ci=0.95):
    """Compute bootstrap confidence interval."""
    boot_stats = []
    data = np.array(data, dtype=float)
    for _ in range(n_bootstrap):
        sample = np.random.choice(data, size=len(data), replace=True)
        boot_stats.append(stat_fn(sample))
    lower = float(np.percentile(boot_stats, (1 - ci) / 2 * 100))
    upper = float(np.percentile(boot_stats, (1 + ci) / 2 * 100))
    return lower, upper

# Build comparison table
comparison = []
for method, results in all_results.items():
    rewards = [r["total_reward"] for r in results]
    successes = [float(r["success"]) for r in results]

    rew_lo, rew_hi = bootstrap_ci(rewards, np.mean, CFG["bootstrap_n"], CFG["ci_level"])
    suc_lo, suc_hi = bootstrap_ci(successes, np.mean, CFG["bootstrap_n"], CFG["ci_level"])

    comparison.append({
        "method":           method,
        "mean_reward":      float(np.mean(rewards)),
        "std_reward":       float(np.std(rewards)),
        "ci95_reward_lo":   rew_lo,
        "ci95_reward_hi":   rew_hi,
        "success_rate":     float(np.mean(successes)),
        "ci95_success_lo":  suc_lo,
        "ci95_success_hi":  suc_hi,
        "mean_steps":       float(np.mean([r["steps"] for r in results])),
    })

comp_df = pd.DataFrame(comparison)
comp_df.to_csv(ARTIFACTS_DIR / "comparison_table.csv", index=False)

print("\nComparison Table (200 episodes, 50K bootstrap):")
print(comp_df.to_string(index=False))


## Step 4b — Welch's t-test & Cohen's d (Pairwise)

- **Welch's t-test**: `scipy.stats.ttest_ind(equal_var=False)` — does not
  assume equal variance.
- **Cohen's d**: standardized effect size = (M₁ − M₂) / pooled SD.


In [ ]:
def cohens_d(x, y):
    """Cohen's d effect size (pooled SD)."""
    nx, ny = len(x), len(y)
    var_x, var_y = np.var(x, ddof=1), np.var(y, ddof=1)
    pooled_sd = np.sqrt(((nx - 1) * var_x + (ny - 1) * var_y) / (nx + ny - 2))
    return (np.mean(x) - np.mean(y)) / pooled_sd if pooled_sd > 0 else 0.0


method_names = list(all_results.keys())
stat_tests = []

for m1, m2 in itertools.combinations(method_names, 2):
    r1 = np.array([r["total_reward"] for r in all_results[m1]])
    r2 = np.array([r["total_reward"] for r in all_results[m2]])

    t_stat, p_value = stats.ttest_ind(r1, r2, equal_var=False)
    d = cohens_d(r1, r2)

    stat_tests.append({
        "comparison":  f"{m1} vs {m2}",
        "t_statistic": float(t_stat),
        "p_value":     float(p_value),
        "cohens_d":    float(d),
        "significant": bool(p_value < 0.05),
        "effect_size": ("large" if abs(d) >= 0.8 else
                        "medium" if abs(d) >= 0.5 else
                        "small" if abs(d) >= 0.2 else "negligible"),
    })

stat_df = pd.DataFrame(stat_tests)
stat_df.to_csv(ARTIFACTS_DIR / "stat_tests.csv", index=False)

with open(ARTIFACTS_DIR / "stat_tests.json", "w") as f:
    json.dump(stat_tests, f, indent=2)

print("\nPairwise Statistical Tests:")
print(stat_df.to_string(index=False))


## Step 5 — Declare Winner


In [ ]:
best_idx = comp_df["mean_reward"].idxmax()
winner = comp_df.loc[best_idx]

best_method = {
    "winner":       str(winner["method"]),
    "mean_reward":  float(winner["mean_reward"]),
    "ci95":         [float(winner["ci95_reward_lo"]), float(winner["ci95_reward_hi"])],
    "success_rate": float(winner["success_rate"]),
    "eval_episodes": CFG["eval_episodes"],
    "bootstrap_n":  CFG["bootstrap_n"],
    "reason":       f"Highest mean reward ({winner['mean_reward']:.4f}) "
                    f"over {CFG['eval_episodes']} episodes with 50K bootstrap CI",
}

with open(ARTIFACTS_DIR / "best_method.json", "w") as f:
    json.dump(best_method, f, indent=2)

print(f"\n{'='*60}")
print(f"  🏆 WINNER: {best_method['winner']}")
print(f"  Mean Reward: {best_method['mean_reward']:.4f} "
      f"[{best_method['ci95'][0]:.4f}, {best_method['ci95'][1]:.4f}]")
print(f"  Success Rate: {best_method['success_rate']:.2%}")
print(f"  → This method will train DishWipe in NB09")
print(f"{'='*60}")


## Step 6 — Comparison Plots


In [ ]:
methods = comp_df["method"].tolist()
color_map = {"PPO": "#FFB74D", "SAC": "#64B5F6", "Residual-SAC": "#81C784"}
colors = [color_map.get(m, "gray") for m in methods]

# Plot A: Reward with CI
fig, ax = plt.subplots(figsize=(8, 5))
means = comp_df["mean_reward"].tolist()
ci_lo = comp_df["ci95_reward_lo"].tolist()
ci_hi = comp_df["ci95_reward_hi"].tolist()
errs = [[m - lo for m, lo in zip(means, ci_lo)],
        [hi - m for m, hi in zip(means, ci_hi)]]
ax.bar(methods, means, yerr=errs, capsize=8, color=colors, edgecolor="black")
ax.set_title("Mean Reward (200 episodes, 95% CI, 50K bootstrap)", fontweight="bold")
ax.set_ylabel("Mean Reward")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(ARTIFACTS_DIR / "comparison_plot.png", dpi=150)
plt.show()

# Plot B: Success Rate
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(methods, comp_df["success_rate"].tolist(), color=colors, edgecolor="black")
ax.set_title("Success Rate (200 episodes)", fontweight="bold")
ax.set_ylabel("Success Rate")
ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(ARTIFACTS_DIR / "success_rate_plot.png", dpi=150)
plt.show()

# Plot C: Reward distribution (violin)
fig, ax = plt.subplots(figsize=(8, 5))
data_violin = [[r["total_reward"] for r in all_results[m]] for m in methods]
parts = ax.violinplot(data_violin, positions=range(len(methods)), showmeans=True)
ax.set_xticks(range(len(methods)))
ax.set_xticklabels(methods)
ax.set_title("Reward Distribution (200 episodes)", fontweight="bold")
ax.set_ylabel("Total Reward")
fig.tight_layout()
fig.savefig(ARTIFACTS_DIR / "reward_distribution.png", dpi=150)
plt.show()

# Plot D: Cohen's d heatmap
fig, ax = plt.subplots(figsize=(6, 5))
d_matrix = np.zeros((len(method_names), len(method_names)))
for test in stat_tests:
    m1, m2 = test["comparison"].split(" vs ")
    i, j = method_names.index(m1), method_names.index(m2)
    d_matrix[i, j] = test["cohens_d"]
    d_matrix[j, i] = -test["cohens_d"]
im = ax.imshow(d_matrix, cmap="RdBu_r", vmin=-2, vmax=2)
ax.set_xticks(range(len(method_names)))
ax.set_xticklabels(method_names, rotation=45, ha="right")
ax.set_yticks(range(len(method_names)))
ax.set_yticklabels(method_names)
for i in range(len(method_names)):
    for j in range(len(method_names)):
        ax.text(j, i, f"{d_matrix[i,j]:.2f}", ha="center", va="center", fontsize=10)
ax.set_title("Cohen's d Effect Size (pairwise)", fontweight="bold")
fig.colorbar(im)
fig.tight_layout()
fig.savefig(ARTIFACTS_DIR / "cohens_d_heatmap.png", dpi=150)
plt.show()

print("✅ All plots saved")


## Cleanup


In [ ]:
print("✅ NB08 Evaluation Complete (RTX 5090 spec)")
print(f"Winner: {best_method['winner']} → will train DishWipe in NB09")


## Artifacts

| File | Description |
|------|-------------|
| `eval_200ep.csv` | Per-episode results for all methods (200 × 3) |
| `comparison_table.csv` | Summary statistics with 95% CI |
| `stat_tests.csv` | Welch's t-test + Cohen's d (pairwise) |
| `stat_tests.json` | Same in JSON format |
| `comparison_plot.png` | Bar chart with 95% CI |
| `success_rate_plot.png` | Success rate comparison |
| `reward_distribution.png` | Violin plot of reward distributions |
| `cohens_d_heatmap.png` | Cohen's d effect size heatmap |
| `best_method.json` | Winner declaration + reason |

## RTX 5090 Evaluation Notes

- **200 episodes** (was 100) — more statistical power
- **50,000 bootstrap resamples** (was 10,000) — tighter CI
- **Welch's t-test** — pairwise, no equal-variance assumption
- **Cohen's d** — standardized effect size (small/medium/large)
- Deterministic eval with `seeds 0-199` → fully reproducible
- `best_method.json` is consumed by NB09
